> **Corrección de firma:** esta revisión corrige la construcción de las firmas de configuración. Se usan `application` y `application_version` ya resueltos y las firmas se crean después de resolver `nodes`, `processes`, `ppn`, `filesystem`, `stripe_size_bytes`, `stripe_count` y `operation_mode`.


## Versión maestra acumulativa

Esta versión integra en un único proyecto:

- construcción dinámica de rutas por aplicación/campaña;
- adaptadores DLIO v1, DLIO actual y DeepGalaxy;
- firmas semánticas de aplicación, dataset y experimento;
- validación de ejecución, instrumentación y metadatos;
- extracción genérica de métricas científicas de aplicación;
- estado de cobertura de métricas: `AVAILABLE`, `PARTIAL`, `UNAVAILABLE`;
- salida `application_metrics_<clasificación>.csv`;
- soporte DeepGalaxy para `operation_mode=read` y métricas de entrenamiento.


# DeepTuneIO-Indexer v12

Versión genérica para indexar ejecuciones HPC de cualquier aplicación, con o sin Darshan.

Estados independientes:
- `execution_status`
- `instrumentation_status`
- `metadata_status`

Esta revisión conserva la **selección dinámica de aplicación/campaña** y añade la extracción genérica de **métricas científicas de la aplicación** mediante los adaptadores.


In [1]:
pwd

'F:\\NoteBooks\\CORA_DR_UAB\\Article_01_Parallel_IO_Analysis\\Scripts\\DeepTuneIO\\07_Indexer'

In [2]:
from pathlib import Path
import sys, json
import pandas as pd

# Carpeta que contiene el módulo principal y el paquete deeptuneio.
PROJECT_DIR = Path('./Module').resolve()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from DeepTuneIO_Indexer_v12 import DeepTuneIOIndexer, write_outputs
from deeptuneio.adapters import AdapterRegistry

print('Proyecto:', PROJECT_DIR)
print('Módulo v10 disponible:', (PROJECT_DIR / 'DeepTuneIO_Indexer_v12.py').exists())


Proyecto: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\07_Indexer\Module
Módulo v10 disponible: True


## 1. Seleccionar aplicación, dataset y escenario

La selección utiliza el mismo catálogo `CAMPAIGNS` y los mismos índices `APPLICATION_POS`, `DATASET_POS` y `SCENARIO_POS` empleados por los módulos DXT y Parser.


In [3]:
# ============================================================
# CONFIGURACIÓN GENERAL Y CATÁLOGO DE CAMPAÑAS
# ============================================================

RESOURCE_PROJECT = "RES-DATA-2022-1-0014"

PROJECT_ROOT = Path(r"F:\\NoteBooks\\CORA_DR_UAB").resolve()
ARTICLE_ID = "Article_01_Parallel_IO_Analysis"
ARTICLE_ROOT = PROJECT_ROOT / ARTICLE_ID

# Catálogo común con los módulos de procesamiento.
CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": [
                "lustre_1ost",
                "lustre_2ost",
                "lustre_4ost",
                "nfs_SnGPU_e1",
            ],
            "file_format": ".hdf5",
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".h5",
        },
        "NPZ_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".npz",
        },
        "TFRecord_256kts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
        },
        "TFRecord_1mts_64bs": {
            "scenarios": ["lustre_ost"],
            "file_format": ".tfrecords",
        },
    },
}

# ------------------------------------------------------------
# Cambiar únicamente estos índices para seleccionar otra campaña.
# Los índices empiezan en 1, igual que en Module 06 y Module 07.
# ------------------------------------------------------------
APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 1

APPLICATION_OPTIONS = list(CAMPAIGNS)
if not 1 <= APPLICATION_POS <= len(APPLICATION_OPTIONS):
    raise ValueError("APPLICATION_POS fuera de rango.")
APPLICATION = APPLICATION_OPTIONS[APPLICATION_POS - 1]

DATASET_OPTIONS = list(CAMPAIGNS[APPLICATION])
if not 1 <= DATASET_POS <= len(DATASET_OPTIONS):
    raise ValueError("DATASET_POS fuera de rango.")
DATASET_GROUP = DATASET_OPTIONS[DATASET_POS - 1]

SCENARIO_OPTIONS = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"]
if not 1 <= SCENARIO_POS <= len(SCENARIO_OPTIONS):
    raise ValueError("SCENARIO_POS fuera de rango.")
SCENARIO = SCENARIO_OPTIONS[SCENARIO_POS - 1]

FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]

print("Project root  :", PROJECT_ROOT)
print("Article root  :", ARTICLE_ROOT)
print("Application   :", APPLICATION)
print("Dataset group :", DATASET_GROUP)
print("Scenario      :", SCENARIO)
print("File format   :", FILE_FORMAT)


Project root  : F:\NoteBooks\CORA_DR_UAB
Article root  : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis
Application   : DeepGalaxy
Dataset group : DG_bw512_f3c5x64
Scenario      : lustre_1ost
File format   : .hdf5


## 2. Construir la raíz de la campaña

La ruta sigue la jerarquía común de DeepTuneIO:

`<PROJECT_ROOT>/<ARTICLE_ID>/Data/<APPLICATION>/<DATASET_GROUP>/<SCENARIO>/`.


In [4]:
# ============================================================
# CONSTRUCCIÓN UNIFICADA DE LA RUTA DE CAMPAÑA
# ============================================================

CAMPAIGN_ROOT = (
    ARTICLE_ROOT
    / "Data"
    / APPLICATION
    / DATASET_GROUP
    / SCENARIO
).resolve()

# El Indexer examina recursivamente la campaña completa porque sus fuentes
# pueden encontrarse en diferentes subdirectorios:
#   - Slurm stdout/stderr
#   - SEFF
#   - Darshan Parser
#   - Darshan DXT
#   - Darshan PERF
INPUT_DIR = CAMPAIGN_ROOT

# Las salidas del Indexer se mantienen separadas de las fuentes y de los
# resultados científicos de Parser/DXT/PERF.
OUTPUT_DIR = CAMPAIGN_ROOT / "results" / "Indexer"

print("Campaign root :", CAMPAIGN_ROOT)
print("Input dir     :", INPUT_DIR)
print("Output dir    :", OUTPUT_DIR)
print("Campaign exists:", CAMPAIGN_ROOT.exists())

if not CAMPAIGN_ROOT.exists():
    raise FileNotFoundError(
        f"No existe la campaña seleccionada: {CAMPAIGN_ROOT}"
    )


Campaign root : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost
Input dir     : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost
Output dir    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer
Campaign exists: True


In [5]:
# ============================================================
# INSPECCIÓN DE FUENTES DISPONIBLES EN LA CAMPAÑA
# ============================================================

# Esta celda no impone una ubicación física concreta para cada evidencia.
# Solo comprueba, de forma recursiva, qué fuentes puede encontrar el Indexer.

def count_matching(root: Path, predicate):
    return sum(1 for p in root.rglob("*") if p.is_file() and predicate(p))

source_counts = {
    "stdout (.out)": count_matching(INPUT_DIR, lambda p: p.suffix.lower() == ".out"),
    "stderr (.err)": count_matching(INPUT_DIR, lambda p: p.suffix.lower() == ".err"),
    "SEFF txt": count_matching(
        INPUT_DIR,
        lambda p: p.suffix.lower() == ".txt" and "seff" in p.name.lower()
    ),
    "Parser txt": count_matching(
        INPUT_DIR,
        lambda p: p.suffix.lower() == ".txt" and "parser" in p.name.lower()
    ),
    "DXT txt": count_matching(
        INPUT_DIR,
        lambda p: p.suffix.lower() == ".txt" and "_dxt" in p.name.lower()
    ),
    "PERF txt": count_matching(
        INPUT_DIR,
        lambda p: p.suffix.lower() == ".txt" and "_perf" in p.name.lower()
    ),
}

for source, count in source_counts.items():
    print(f"{source:15s}: {count}")


stdout (.out)  : 39
stderr (.err)  : 39
SEFF txt       : 39
Parser txt     : 39
DXT txt        : 39
PERF txt       : 39


## 2. Construir índice

## Resultados de entrenamiento

La v12 conserva la serie completa en `application_metrics_*.csv` y añade `training_summary_*.csv`, con una fila por JobID validado. Este resumen incluye estado de entrenamiento, épocas esperadas/completadas, métricas finales y los mínimos/máximos de `loss`, `validation_loss`, `accuracy` y `validation_accuracy` cuando la aplicación los expone mediante su adaptador.


In [6]:
indexer = DeepTuneIOIndexer(resource_project=RESOURCE_PROJECT)
records, issues = indexer.run(INPUT_DIR)
outputs = write_outputs(records, issues, OUTPUT_DIR)

print(f'Experimentos indexados: {len(records)}')
for name, path in outputs.items():
    print(f'{name}: {path.resolve()}')


Experimentos indexados: 39
csv: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\experiment_index_hdf5.csv
incomplete_csv: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\incomplete_experiments_hdf5.csv
json: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\experiment_index_master_hdf5.json
validation_report: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\validation_report_hdf5.txt
experiment_matrix: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\experiment_matrix_hdf5.csv
campaign_coverage: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\Lustre_1ost\results\Indexer\campaign_coverage_hdf5.json
applica

## 3. Inspeccionar resultados

In [6]:
df = pd.read_csv(outputs['csv'])
df


,experiment_id,sequence_id,application,application_version,application_version_source,application_version_confidence,configuration_source,job_id,platform,resource_project,...,perf_file,seff_file,stdout_file,stderr_file,paper_figures,paper_tables,execution_status,instrumentation_status,metadata_status,validation_notes
0,DTIO_4935078,1,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,4935078,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\1_DG_N1p4e1_EfficientNetB4_outputbw512hdf...,Seff\1_DeepGalaxy_lustre_1ost_bw512_4935078_hd...,Slurm\1_DG_L1ost_e1_4935078.out,Slurm\1_DG_L1ost_e1_4935078.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
1,DTIO_4969259,21,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,4969259,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\21_DG_N1p4e1_EfficientNetB4_outputbw512hd...,Seff\21_DG_L1ost_bw512_4969259_hdf5_seff_JSC24...,Slurm\21_DG_L1ost_e1_4969259.out,Slurm\21_DG_L1ost_e1_4969259.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
2,DTIO_4998171,7,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,4998171,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\7_DG_N2p8e1_EfficientNetB4_outputbw512hdf...,Seff\7_DeepGalaxy_lustre_1ost_bw512_4998171_hd...,Slurm\7_DG_L1ost_e1_4998171.out,Slurm\7_DG_L1ost_e1_4998171.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
3,DTIO_5004132,27,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5004132,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\27_DG_N4p16e1_EfficientNetB4_outputbw512h...,Seff\27_DeepGalaxy_lustre_1ost_bw512_5004132_h...,Slurm\27_DG_L1ost_e1_5004132.out,Slurm\27_DG_L1ost_e1_5004132.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
4,DTIO_5005753,31,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5005753,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\31_DG_N8p32e1_EfficientNetB4_outputbw512h...,Seff\31_DeepGalaxy_lustre_1ost_bw512_5005753_h...,Slurm\31_DG_L1ost_e1_5005753.out,Slurm\31_DG_L1ost_e1_5005753.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
5,DTIO_5006747,35,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5006747,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\35_DG_N12p48e1_EfficientNetB4_outputbw512...,Seff\35_DeepGalaxy_lustre_1ost_bw512_5006747_h...,Slurm\35_DG_L1ost_e1_5006747.out,Slurm\35_DG_L1ost_e1_5006747.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
6,DTIO_5136097,24,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5136097,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\24_DG_N2p8e1_EfficientNetB4_outputbw512hd...,Seff\24_DG_L1ost_bw512_5136097_hdf5_seff_JSC24...,Slurm\24_DG_L1ost_e1_5136097.out,Slurm\24_DG_L1ost_e1_5136097.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
7,DTIO_5145754,25,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5145754,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\25_DG_N2p8e1_EfficientNetB4_outputbw512hd...,Seff\25_DeepGalaxy_lustre_1ost_bw512_5145754_h...,Slurm\25_DG_L1ost_e1_5145754.out,Slurm\25_DG_L1ost_e1_5145754.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
8,DTIO_5147852,4,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5147852,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\4_DG_N2p8e1_EfficientNetB4_outputbw512hdf...,Seff\4_DeepGalaxy_lustre_1ost_bw512_5147852_hd...,Slurm\4_DG_L1ost_e1_5147852.out,Slurm\4_DG_L1ost_e1_5147852.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages detected.
9,DTIO_5164453,5,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,deepgalaxy_cli,5164453,FinisTerrae III,RES-DATA-2022-1-0014,...,Perf\5_DG_N2p8e1_EfficientNetB4_outputbw512hdf...,Seff\5_DeepGalaxy_lustre_1ost_bw512_5164453_hd...,Slurm\5_DG_L1ost_e1_5164453.out,Slurm\5_DG_L1ost_e1_5164453.err,NaN,NaN,PASS,COMPLETE,PASS,[INFO] stderr: 2 warning-like messages det

## 3.1. Métricas científicas de la aplicación

Si el adaptador de la aplicación implementa `parse_application_metrics()` o `parse_application_metric_series()`, DeepTuneIO genera automáticamente `application_metrics_<clasificación>.csv`. El núcleo no presupone nombres concretos como `loss`, `accuracy`, `throughput` o `AUC`; cada adaptador define cómo extraer y normalizar sus resultados.


In [7]:
# El fichero solo existe cuando la aplicación expone métricas científicas.
if 'application_metrics' in outputs:
    application_metrics_df = pd.read_csv(outputs['application_metrics'])
    print('Métricas de aplicación:', outputs['application_metrics'])
    display(application_metrics_df)
else:
    application_metrics_df = pd.DataFrame()
    print('La aplicación/campaña seleccionada no generó métricas científicas adicionales.')


Métricas de aplicación: F:\NoteBooks\Journal_SuperComputing25\Traces\DeepGalaxy\Input\DG_bw512_f3c5x64\lustre_1ost\deeptuneio_index_v10\application_metrics_hdf5.csv


,experiment_id,job_id,application,application_version,step_type,step,metric_name,metric_value,native_metric_name,metric_namespace,source
0,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,loss,6.7458,loss,training,application_stdout
1,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,accuracy,0.0046,sparse_categorical_accuracy,training,application_stdout
2,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,validation_loss,6.2655,val_loss,training,application_stdout
3,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,validation_accuracy,0.0128,val_sparse_categorical_accuracy,training,application_stdout
4,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,training_time_s,25120.0000,keras_epoch_elapsed_seconds,training,application_stdout
...,...,...,...,...,...,...,...,...,...,...,...
267,DTIO_8850850,8850850,DeepGalaxy,legacy,epoch,1,validation_loss,11.3533,val_loss,training,application_stdout
268,DTIO_8850850,8850850,DeepGalaxy,legacy,epoch,1,validation_accuracy,0.0022,val_sparse_categorical_accuracy,training,application_stdout
269,DTIO_8850850,8850850,DeepGalaxy,legacy,epoch,1,training_time_s,4006.0000,keras_epoch_elapsed_seconds,training,application_stdout
270,DTIO_8850850,8850850,DeepGalaxy,legacy,epoch,1,step_time_s,17.0000,keras_seconds_per_step,training,application_stdout


### Resumen de métricas por ejecución

Para aplicaciones de entrenamiento, como DeepGalaxy, esta vista permite comprobar los resultados finales por `JobID` sin incorporarlos a las firmas de configuración. Las métricas siguen siendo **resultados del experimento**, no parámetros de configuración.


In [8]:
if not application_metrics_df.empty:
    metrics_summary_df = (
        application_metrics_df
        .pivot_table(
            index=['experiment_id','job_id','application','application_version','step_type','step'],
            columns='metric_name',
            values='metric_value',
            aggfunc='first'
        )
        .reset_index()
    )
    display(metrics_summary_df)
else:
    print('No hay métricas de aplicación para resumir.')


metric_name,experiment_id,job_id,application,application_version,step_type,step,accuracy,loss,step_time_s,steps,training_time_s,validation_accuracy,validation_loss
0,DTIO_4935078,4935078,DeepGalaxy,legacy,epoch,1,0.0046,6.7458,14.0,1789.0,25120.0,0.012800,6.2655
1,DTIO_4969259,4969259,DeepGalaxy,legacy,epoch,1,0.0044,6.7387,14.0,1789.0,24887.0,0.010100,6.4730
2,DTIO_4998171,4998171,DeepGalaxy,legacy,epoch,1,0.0077,6.5260,14.0,895.0,12814.0,0.003200,11.7848
3,DTIO_5004132,5004132,DeepGalaxy,legacy,epoch,1,0.0042,6.6145,16.0,447.0,7175.0,0.003100,17.5969
4,DTIO_5005753,5005753,DeepGalaxy,legacy,epoch,1,0.0033,6.8034,27.0,224.0,6345.0,0.003100,7.1425
5,DTIO_5006747,5006747,DeepGalaxy,legacy,epoch,1,0.0036,6.8478,19.0,149.0,2934.0,0.002800,15.9294
6,DTIO_5136097,5136097,DeepGalaxy,legacy,epoch,1,0.0065,6.5814,18.0,895.0,16487.0,0.024400,5.8538
7,DTIO_5145754,5145754,DeepGalaxy,legacy,epoch,1,0.0056,6.5992,16.0,895.0,14541.0,0.013100,6.3585
8,DTIO_5147852,5147852,DeepGalaxy,legacy,epoch,1,0.0075,6.5608,16.0,895.0,14175.0,0.020400,7.3447
9,DTIO_5164453,5164453,DeepGalaxy,legacy,epoch,1,0.0054,6.6159,16.0,895.0,14730.0,NaN,6.5118


### Ejemplo: métricas de entrenamiento normalizadas

Cuando el adaptador proporciona nombres normalizados, pueden visualizarse de forma uniforme. Esta celda no es obligatoria y funciona solo si esas métricas existen.


In [9]:
if not application_metrics_df.empty:
    wanted = {
        'loss', 'accuracy', 'validation_loss', 'validation_accuracy',
        'training_time_s', 'step_time_s', 'steps'
    }
    available = sorted(set(application_metrics_df['metric_name']) & wanted)
    if available:
        display(
            application_metrics_df[
                application_metrics_df['metric_name'].isin(available)
            ][[
                'experiment_id','job_id','step_type','step',
                'metric_name','metric_value','native_metric_name','source'
            ]]
        )
    else:
        print('La aplicación seleccionada no expone estas métricas normalizadas.')


,experiment_id,job_id,step_type,step,metric_name,metric_value,native_metric_name,source
0,DTIO_4935078,4935078,epoch,1,loss,6.7458,loss,application_stdout
1,DTIO_4935078,4935078,epoch,1,accuracy,0.0046,sparse_categorical_accuracy,application_stdout
2,DTIO_4935078,4935078,epoch,1,validation_loss,6.2655,val_loss,application_stdout
3,DTIO_4935078,4935078,epoch,1,validation_accuracy,0.0128,val_sparse_categorical_accuracy,application_stdout
4,DTIO_4935078,4935078,epoch,1,training_time_s,25120.0000,keras_epoch_elapsed_seconds,application_stdout
...,...,...,...,...,...,...,...,...
267,DTIO_8850850,8850850,epoch,1,validation_loss,11.3533,val_loss,application_stdout
268,DTIO_8850850,8850850,epoch,1,validation_accuracy,0.0022,val_sparse_categorical_accuracy,application_stdout
269,DTIO_8850850,8850850,epoch,1,training_time_s,4006.0000,keras_epoch_elapsed_seconds,application_stdout
270,DTIO_8850850,8850850,epoch,1,step_time_s,17.0000,keras_seconds_per_step,application_stdout


## 4. Estados de ejecución e instrumentación

In [10]:
cols = [
    'experiment_id','job_id','application','application_version',
    'application_version_source','application_version_confidence',
    'job_state','exit_code',
    'execution_status','instrumentation_status','metadata_status',
    'stdout_available','stderr_available','darshan_available'
]
df[cols]


,experiment_id,job_id,application,application_version,application_version_source,application_version_confidence,job_state,exit_code,execution_status,instrumentation_status,metadata_status,stdout_available,stderr_available,darshan_available
0,DTIO_4935078,4935078,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
1,DTIO_4969259,4969259,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
2,DTIO_4998171,4998171,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
3,DTIO_5004132,5004132,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
4,DTIO_5005753,5005753,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
5,DTIO_5006747,5006747,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
6,DTIO_5136097,5136097,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
7,DTIO_5145754,5145754,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
8,DTIO_5147852,5147852,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True
9,DTIO_5164453,5164453,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,COMPLETED,0,PASS,COMPLETE,PASS,True,True,True


## 5. Ver ejecuciones sin Darshan

In [11]:
df[df['instrumentation_status'] == 'MISSING'][[
    'experiment_id','job_id','application','execution_status',
    'stdout_file','stderr_file'
]]


,experiment_id,job_id,application,execution_status,stdout_file,stderr_file


## 6. Ver warnings / errores detectados

In [12]:
df[(df['warning_count'] > 0) | (df['error_count'] > 0)][[
    'experiment_id','job_id','warning_count','error_count',
    'execution_status','stderr_file'
]]


,experiment_id,job_id,warning_count,error_count,execution_status,stderr_file
0,DTIO_4935078,4935078,2,0,PASS,Slurm\1_DG_L1ost_e1_4935078.err
1,DTIO_4969259,4969259,2,0,PASS,Slurm\21_DG_L1ost_e1_4969259.err
2,DTIO_4998171,4998171,2,0,PASS,Slurm\7_DG_L1ost_e1_4998171.err
3,DTIO_5004132,5004132,2,0,PASS,Slurm\27_DG_L1ost_e1_5004132.err
4,DTIO_5005753,5005753,2,0,PASS,Slurm\31_DG_L1ost_e1_5005753.err
5,DTIO_5006747,5006747,2,0,PASS,Slurm\35_DG_L1ost_e1_5006747.err
6,DTIO_5136097,5136097,2,0,PASS,Slurm\24_DG_L1ost_e1_5136097.err
7,DTIO_5145754,5145754,2,0,PASS,Slurm\25_DG_L1ost_e1_5145754.err
8,DTIO_5147852,5147852,2,0,PASS,Slurm\4_DG_L1ost_e1_5147852.err
9,DTIO_5164453,5164453,2,0,PASS,Slurm\5_DG_L1ost_e1_5164453.err


## 7. Trazabilidad con el artículo

Los campos `paper_figures` y `paper_tables` se completan explícitamente después de validar qué ejecuciones sustentan cada resultado.

## 8. Resumen semántico de diagnósticos

Las ocurrencias repetidas por proceso se conservan, pero también se agrupan por tipo diagnóstico.

In [13]:
df[[
    'experiment_id','job_id','execution_status',
    'warning_count','error_count','diagnostic_occurrences',
    'diagnostic_types','diagnostic_counts'
]]


,experiment_id,job_id,execution_status,warning_count,error_count,diagnostic_occurrences,diagnostic_types,diagnostic_counts
0,DTIO_4935078,4935078,PASS,2,0,12,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
1,DTIO_4969259,4969259,PASS,2,0,12,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
2,DTIO_4998171,4998171,PASS,2,0,24,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
3,DTIO_5004132,5004132,PASS,2,0,48,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
4,DTIO_5005753,5005753,PASS,2,0,100,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
5,DTIO_5006747,5006747,PASS,2,0,145,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
6,DTIO_5136097,5136097,PASS,2,0,30,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
7,DTIO_5145754,5145754,PASS,2,0,25,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
8,DTIO_5147852,5147852,PASS,2,0,24,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."
9,DTIO_5164453,5164453,PASS,2,0,28,"[""generic_warning"", ""tensorrt_unavailable""]","{""cuda_initialization_unavailable"": 0, ""cuda_l..."


## 9. Validar el formato de fichero

El campo universal `file_format` se resuelve por prioridad: adaptador/comando de aplicación, DXT, stdout y finalmente nombre de fichero.

In [14]:
df[[
    'experiment_id','job_id','application','application_version',
    'application_version_source','application_version_confidence',
    'file_format','file_format_source','file_format_confidence',
    'instrumentation_status','metadata_status'
]]


,experiment_id,job_id,application,application_version,application_version_source,application_version_confidence,file_format,file_format_source,file_format_confidence,instrumentation_status,metadata_status
0,DTIO_4935078,4935078,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
1,DTIO_4969259,4969259,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
2,DTIO_4998171,4998171,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
3,DTIO_5004132,5004132,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
4,DTIO_5005753,5005753,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
5,DTIO_5006747,5006747,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
6,DTIO_5136097,5136097,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
7,DTIO_5145754,5145754,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
8,DTIO_5147852,5147852,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS
9,DTIO_5164453,5164453,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,hdf5,application_command,HIGH,COMPLETE,PASS


### Comprobar formatos detectados

Para DLIO con Darshan, `file_format_source` debería ser `application_command` porque `# exe` contiene `-f`.

In [15]:
df.groupby(['file_format','file_format_source','file_format_confidence'], dropna=False).size().reset_index(name='executions')


,file_format,file_format_source,file_format_confidence,executions
0,hdf5,application_command,HIGH,39


## 10. Configuración experimental normalizada

DeepTuneIO v12 separa parámetros universales, parámetros comunes al formato y parámetros específicos de la aplicación.

In [16]:
normalized_cols = [
    'experiment_id','job_id','application','application_version','file_format',
    'batch_size','number_files','samples_per_file','record_length_bytes','epochs',
    'transfer_size_bytes','transfer_size_label',
    'chunking_enabled','chunk_size_bytes','compression_type','compression_level',
    'nodes','processes','ppn','filesystem','stripe_size_bytes','stripe_count',
    'configuration_source','application_configuration_signature','dataset_configuration_signature','experiment_configuration_signature'
]
df[normalized_cols]


,experiment_id,job_id,application,application_version,file_format,batch_size,number_files,samples_per_file,record_length_bytes,epochs,...,nodes,processes,ppn,filesystem,stripe_size_bytes,stripe_count,configuration_source,application_configuration_signature,dataset_configuration_signature,experiment_configuration_signature
0,DTIO_4935078,4935078,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,1,4,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
1,DTIO_4969259,4969259,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,1,4,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
2,DTIO_4998171,4998171,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,2,8,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
3,DTIO_5004132,5004132,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,4,16,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
4,DTIO_5005753,5005753,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,8,32,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
5,DTIO_5006747,5006747,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,12,48,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
6,DTIO_5136097,5136097,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,2,8,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
7,DTIO_5145754,5145754,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,2,8,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
8,DTIO_5147852,5147852,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,2,8,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
9,DTIO_5164453,5164453,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,1,...,2,8,4,lustre,1048576,1,deepgalaxy_cli,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."


### TFRecord por tamaño de transferencia

Permite distinguir, por ejemplo, 256 KiB y 1 MiB sin alterar `file_format=tfrecord`.

In [17]:
if 'transfer_size_label' in df.columns:
    display(df[df['file_format'] == 'tfrecord'][[
        'experiment_id','job_id','file_format','batch_size',
        'transfer_size_bytes','transfer_size_label','metadata_status'
    ]])


,experiment_id,job_id,file_format,batch_size,transfer_size_bytes,transfer_size_label,metadata_status


### HDF5: parámetros comunes del formato

DeepTuneIO v12 normaliza, cuando están disponibles, los parámetros HDF5 comunes entre aplicaciones:

- `chunking_enabled`
- `chunk_size_bytes`
- `compression_type`
- `compression_level`

En DLIO v1 se obtienen de `-ec`, `-cs`, `-co` y `-cl`. En DLIO 2.x/current se obtienen de la configuración YAML/Hydra equivalente. Los parámetros originales permanecen además en `application_parameters`.


## 11. Separación entre experimentos validados e incompletos

El CSV principal contiene únicamente experimentos con `PASS / COMPLETE / PASS`. Las demás ejecuciones se conservan en un CSV separado y en el JSON maestro.

In [18]:
validated_df = pd.read_csv(outputs['csv'])
incomplete_df = pd.read_csv(outputs['incomplete_csv'])

print('Experimentos validados:', len(validated_df))
print('Ejecuciones incompletas/no validadas:', len(incomplete_df))


Experimentos validados: 39
Ejecuciones incompletas/no validadas: 0


### Experimentos validados

In [19]:
validated_df[[
    'experiment_id','job_id','application','application_version',
    'file_format','batch_size','number_files','samples_per_file',
    'record_length_bytes','transfer_size_label',
    'nodes','processes','execution_status',
    'instrumentation_status','metadata_status'
]]


,experiment_id,job_id,application,application_version,file_format,batch_size,number_files,samples_per_file,record_length_bytes,transfer_size_label,nodes,processes,execution_status,instrumentation_status,metadata_status
0,DTIO_4935078,4935078,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,1,4,PASS,COMPLETE,PASS
1,DTIO_4969259,4969259,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,1,4,PASS,COMPLETE,PASS
2,DTIO_4998171,4998171,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,2,8,PASS,COMPLETE,PASS
3,DTIO_5004132,5004132,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,4,16,PASS,COMPLETE,PASS
4,DTIO_5005753,5005753,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,8,32,PASS,COMPLETE,PASS
5,DTIO_5006747,5006747,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,12,48,PASS,COMPLETE,PASS
6,DTIO_5136097,5136097,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,2,8,PASS,COMPLETE,PASS
7,DTIO_5145754,5145754,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,2,8,PASS,COMPLETE,PASS
8,DTIO_5147852,5147852,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,2,8,PASS,COMPLETE,PASS
9,DTIO_5164453,5164453,DeepGalaxy,legacy,hdf5,4,NaN,NaN,NaN,NaN,2,8,PASS,COMPLETE,PASS


### Ejecuciones incompletas o no validadas

No se eliminan: se preservan para trazabilidad y pueden reclasificarse si se recuperan datos faltantes.

In [20]:
incomplete_df[[
    'experiment_id','job_id','application','file_format',
    'execution_status','instrumentation_status','metadata_status',
    'incomplete_reason','stdout_file','stderr_file'
]]


,experiment_id,job_id,application,file_format,execution_status,instrumentation_status,metadata_status,incomplete_reason,stdout_file,stderr_file


## 12. Recuperación conservadora de metadatos de campaña

DeepTuneIO v12 puede completar metadatos faltantes en ejecuciones incompletas usando únicamente valores invariantes y no ambiguos observados en la misma campaña. Esta recuperación **no convierte** una ejecución incompleta en experimento validado.

In [21]:
master = json.loads(Path(outputs['json']).read_text(encoding='utf-8'))
recovered = [r for r in master['records'] if r.get('metadata_recovery')]

print('Ejecuciones con metadatos recuperados:', len(recovered))

pd.DataFrame(recovered)[[
    'experiment_id','job_id','application','application_version',
    'file_format','file_format_source','file_format_confidence',
    'batch_size','record_length_bytes',
    'transfer_size_bytes','transfer_size_label','metadata_recovery',
    'scientific_validation_status','incomplete_reason'
]] if recovered else pd.DataFrame()


Ejecuciones con metadatos recuperados: 0


""


## 13. Matriz experimental

DeepTuneIO v12 construye automáticamente una matriz de configuraciones a partir **solo de los experimentos validados**. Las ejecuciones con la misma configuración se agrupan como réplicas.

In [22]:
matrix_df = pd.read_csv(outputs['experiment_matrix'])
coverage = json.loads(Path(outputs['campaign_coverage']).read_text(encoding='utf-8'))

print('Experimentos validados:', coverage['validated_experiments'])
print('Configuraciones únicas:', coverage['unique_configurations'])
print('Configuraciones con réplicas:', coverage['replicated_configurations'])
matrix_df


Experimentos validados: 39
Configuraciones únicas: 12
Configuraciones con réplicas: 12


,configuration_id,application,application_version,file_format,nodes,processes,ppn,batch_size,number_files,samples_per_file,...,application_configuration_signature,dataset_configuration_signature,experiment_configuration_signature,replicate_count,experiment_ids,job_ids,runtime_s_mean,io_time_s_mean,bandwidth_mib_s_mean,iops_mean
0,CFG_0001,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,...,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,DTIO_4935078;DTIO_5196039;DTIO_5350177,4935078;5196039;5350177,26808.028333,11.212027,187.432005,25175.149399
1,CFG_0002,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,...,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,DTIO_4969259;DTIO_5186976;DTIO_5317272,4969259;5186976;5317272,27959.109533,30.128272,523.429290,8814.680312
2,CFG_0003,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,...,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,DTIO_5006747;DTIO_5173520;DTIO_5347797,5006747;5173520;5347797,3168.795767,16.521763,8338.051780,170114.580400
3,CFG_0004,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,...,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,DTIO_5171929;DTIO_5172393;DTIO_8849686,5171929;5172393;8849686,3291.244967,5.266927,417.787239,119185.423377
4,CFG_0005,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,...,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,DTIO_5333215;DTIO_8787065;DTIO_8806303,5333215;8787065;8806303,2792.903667,5.795052,400.501885,117389.903548
5,CFG_0006,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,...,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",2,DTIO_5366451;DTIO_5369484,5366451;5369484,2591.076000,16.355838,9827.040011,256435.646348
6,CFG_0007,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,...,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",4,DTIO_4998171;DTIO_5147852;DTIO_5164453;DTIO_53...,4998171;5147852;5164453;5352783,14529.803975,8.239657,243.770776,63265.440349
7,CFG_0008,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,...,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,DTIO_5136097;DTIO_5145754;DTIO_5353861,5136097;5145754;5353861,15971.697400,29.156890,1070.953256,18493.003794
8,CFG_0009,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,...,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",4,DTIO_5004132;DTIO_5185397;DTIO_5189973;DTIO_87...,5004132;5185397;5189973;8773752,7400.496300,24.110563,2663.606748,45533.232064
9,CFG_0010,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,...,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,DTIO_5184753;DTIO_8784568;DTIO_8810008,5184753;8784568;8810008,7159.264033,6.284354,322.934486,85592.891051


### Réplicas detectadas

Una réplica comparte la misma configuración normalizada: aplicación, formato, escala, batch size, sistema de archivos, striping y parámetros comunes del formato.

In [23]:
matrix_df[matrix_df['replicate_count'] > 1][[
    'configuration_id','application','application_version',
    'file_format','nodes','processes','ppn','batch_size',
    'number_files','samples_per_file','record_length_bytes',
    'transfer_size_bytes','chunking_enabled','chunk_size_bytes',
    'compression_type','compression_level',
    'stripe_count','replicate_count','job_ids'
]]


,configuration_id,application,application_version,file_format,nodes,processes,ppn,batch_size,number_files,samples_per_file,record_length_bytes,transfer_size_bytes,chunking_enabled,chunk_size_bytes,compression_type,compression_level,stripe_count,replicate_count,job_ids
0,CFG_0001,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,4935078;5196039;5350177
1,CFG_0002,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,4969259;5186976;5317272
2,CFG_0003,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,5006747;5173520;5347797
3,CFG_0004,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,5171929;5172393;8849686
4,CFG_0005,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,5333215;8787065;8806303
5,CFG_0006,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2,5366451;5369484
6,CFG_0007,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,4,4998171;5147852;5164453;5352783
7,CFG_0008,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,5136097;5145754;5353861
8,CFG_0009,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,4,5004132;5185397;5189973;8773752
9,CFG_0010,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,3,5184753;8784568;8810008


### Cobertura de campaña

DeepTuneIO v12 no inventa configuraciones faltantes suponiendo un diseño factorial. El JSON de cobertura registra los valores observados; los huecos reales se determinarán cuando se proporcione o construya el diseño experimental esperado.

In [ ]:
coverage


## 14. Configuración reproducible del dataset

`number_files` y `samples_per_file` forman parte de la clave de configuración reproducible. Dos ejecuciones solo se consideran réplicas si coinciden también estos parámetros, junto con la versión de la aplicación, la configuración de paralelismo, almacenamiento y los parámetros normalizados del formato.

DeepTuneIO conserva además `application_parameters`, `normalized_parameters`, `configuration_parameters` y `configuration_signature`, por lo que los parámetros no normalizados o propios de una aplicación no se pierden.


In [24]:
matrix_df[[
    'configuration_id','application','application_version','file_format',
    'nodes','processes','ppn','batch_size',
    'number_files','samples_per_file','record_length_bytes',
    'transfer_size_bytes','chunking_enabled','chunk_size_bytes',
    'compression_type','compression_level',
    'filesystem','stripe_size_bytes','stripe_count',
    'application_configuration_signature','dataset_configuration_signature','experiment_configuration_signature','replicate_count','job_ids'
]]


,configuration_id,application,application_version,file_format,nodes,processes,ppn,batch_size,number_files,samples_per_file,...,compression_type,compression_level,filesystem,stripe_size_bytes,stripe_count,application_configuration_signature,dataset_configuration_signature,experiment_configuration_signature,replicate_count,job_ids
0,CFG_0001,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,4935078;5196039;5350177
1,CFG_0002,DeepGalaxy,legacy,hdf5,1,4,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,4969259;5186976;5317272
2,CFG_0003,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,5006747;5173520;5347797
3,CFG_0004,DeepGalaxy,legacy,hdf5,12,48,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5171929;5172393;8849686
4,CFG_0005,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5333215;8787065;8806303
5,CFG_0006,DeepGalaxy,legacy,hdf5,16,64,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",2,5366451;5369484
6,CFG_0007,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",4,4998171;5147852;5164453;5352783
7,CFG_0008,DeepGalaxy,legacy,hdf5,2,8,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,5136097;5145754;5353861
8,CFG_0009,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""...",4,5004132;5185397;5189973;8773752
9,CFG_0010,DeepGalaxy,legacy,hdf5,4,16,4,4,NaN,NaN,...,NaN,NaN,lustre,1048576,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5184753;8784568;8810008


## 15. Detección automática de aplicación y versión

DeepTuneIO v12 utiliza adaptadores separados por aplicación y versión. La identidad se determina mediante evidencias del comando, configuración y metadatos disponibles; no se fuerza una versión cuando la evidencia es insuficiente.


In [25]:
df.groupby([
    'application','application_version',
    'application_version_source','application_version_confidence'
], dropna=False).size().reset_index(name='executions')


,application,application_version,application_version_source,application_version_confidence,executions
0,DeepGalaxy,legacy,dg_train_cli_signature,HIGH,39


### Parámetros originales y normalizados

Esto permite verificar que el adaptador de la aplicación no elimina parámetros específicos de la versión.


In [26]:
df[[
    'experiment_id','job_id','application','application_version',
    'application_parameters','normalized_parameters',
    'configuration_parameters','parameter_roles','derived_parameters','application_configuration_signature','dataset_configuration_signature','experiment_configuration_signature'
]].head()


,experiment_id,job_id,application,application_version,application_parameters,normalized_parameters,configuration_parameters,parameter_roles,derived_parameters,application_configuration_signature,dataset_configuration_signature,experiment_configuration_signature
0,DTIO_4935078,4935078,DeepGalaxy,legacy,"{""data_loading_mode"": 0, ""datasets_pattern"": ""...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
1,DTIO_4969259,4969259,DeepGalaxy,legacy,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
2,DTIO_4998171,4998171,DeepGalaxy,legacy,"{""data_loading_mode"": 0, ""datasets_pattern"": ""...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
3,DTIO_5004132,5004132,DeepGalaxy,legacy,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
4,DTIO_5005753,5005753,DeepGalaxy,legacy,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."


## 16. Firmas de configuración

`application_configuration_signature` representa únicamente la configuración propia de la aplicación y su versión. `experiment_configuration_signature` añade escala de ejecución y almacenamiento (`nodes`, `processes`, `ppn`, `filesystem`, `stripe_size_bytes`, `stripe_count`, `operation_mode`). La segunda es la firma reproducible completa.


In [27]:
matrix_df[[
    'configuration_id','nodes','processes','ppn','stripe_count',
    'application_configuration_signature','experiment_configuration_signature',
    'replicate_count','job_ids'
]]


,configuration_id,nodes,processes,ppn,stripe_count,application_configuration_signature,experiment_configuration_signature,replicate_count,job_ids
0,CFG_0001,1,4,4,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,4935078;5196039;5350177
1,CFG_0002,1,4,4,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,4969259;5186976;5317272
2,CFG_0003,12,48,4,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,5006747;5173520;5347797
3,CFG_0004,12,48,4,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5171929;5172393;8849686
4,CFG_0005,16,64,4,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5333215;8787065;8806303
5,CFG_0006,16,64,4,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...",2,5366451;5369484
6,CFG_0007,2,8,4,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...",4,4998171;5147852;5164453;5352783
7,CFG_0008,2,8,4,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...",3,5136097;5145754;5353861
8,CFG_0009,4,16,4,1,"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...",4,5004132;5185397;5189973;8773752
9,CFG_0010,4,16,4,1,"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": ""shared"", ""allow_growth"": ...",3,5184753;8784568;8810008


## 17. Validación semántica de parámetros y firmas

La configuración se separa ahora en tres niveles: aplicación lógica, dataset lógico y experimento reproducible completo. `number_files` y `samples_per_file` se conservan siempre para reproducibilidad, pero DLIO puede marcarlos como `SCALING`; en ese caso no fragmentan la firma lógica del dataset. `total_samples` se deriva únicamente cuando ambos valores están observados.


In [28]:
print('Firmas aplicación:', validated_df['application_configuration_signature'].nunique())
print('Firmas dataset:', validated_df['dataset_configuration_signature'].nunique())
print('Firmas experimento:', validated_df['experiment_configuration_signature'].nunique())

validated_df[[
    'experiment_id','nodes','processes','number_files','samples_per_file',
    'parameter_roles','derived_parameters',
    'application_configuration_signature','dataset_configuration_signature',
    'experiment_configuration_signature'
]].head()


Firmas aplicación: 2
Firmas dataset: 1
Firmas experimento: 12


,experiment_id,nodes,processes,number_files,samples_per_file,parameter_roles,derived_parameters,application_configuration_signature,dataset_configuration_signature,experiment_configuration_signature
0,DTIO_4935078,1,4,NaN,NaN,"{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
1,DTIO_4969259,1,4,NaN,NaN,"{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
2,DTIO_4998171,2,8,NaN,NaN,"{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared"", ""allow_growth"": ...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared"", ""allow_growth"": ..."
3,DTIO_5004132,4,16,NaN,NaN,"{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."
4,DTIO_5005753,8,32,NaN,NaN,"{""access_strategy"": [""APPLICATION""], ""allow_gr...",{},"{""access_strategy"": ""shared_reload_shuffle"", ""...","{""application"": ""DeepGalaxy"", ""application_ver...","{""access_strategy"": ""shared_reload_shuffle"", ""..."


## 18. Parámetros efectivos y valores por defecto de DLIO v1

El adaptador distingue ahora entre parámetros presentes explícitamente en `# exe:` y valores por defecto conocidos de DLIO v1. Los defaults se incorporan a `normalized_parameters` y a las firmas reproducibles, pero su procedencia queda registrada en `parameter_provenance`. Una lista de opciones permitidas no se interpreta automáticamente como valor por defecto.


In [29]:
validated_df[[
    'experiment_id','application','application_version','file_format',
    'application_parameters','normalized_parameters','parameter_provenance'
]].head()


,experiment_id,application,application_version,file_format,application_parameters,normalized_parameters,parameter_provenance
0,DTIO_4935078,DeepGalaxy,legacy,hdf5,"{""data_loading_mode"": 0, ""datasets_pattern"": ""...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": {""confidence"": ""HIGH"", ""so..."
1,DTIO_4969259,DeepGalaxy,legacy,hdf5,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": {""confidence"": ""HIGH"", ""so..."
2,DTIO_4998171,DeepGalaxy,legacy,hdf5,"{""data_loading_mode"": 0, ""datasets_pattern"": ""...","{""access_strategy"": ""shared"", ""allow_growth"": ...","{""access_strategy"": {""confidence"": ""HIGH"", ""so..."
3,DTIO_5004132,DeepGalaxy,legacy,hdf5,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": {""confidence"": ""HIGH"", ""so..."
4,DTIO_5005753,DeepGalaxy,legacy,hdf5,"{""data_loading_mode"": 1, ""datasets_pattern"": ""...","{""access_strategy"": ""shared_reload_shuffle"", ""...","{""access_strategy"": {""confidence"": ""HIGH"", ""so..."
